In [1]:
import os
import sys
sys.path.insert(
    0, os.path.abspath('../../')
)

import json
import yaml

from pathlib import Path
from rich.console import Console
from rich.table import Table

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
root_dir = Path("../../").resolve()
print("Root directory:", root_dir)

Root directory: /home/hgkahng/Workspaces/soft-prompt


In [3]:
N_train = 25_000

## 1. Load Synthetic Data

In [4]:
load_dir = root_dir / "results/imdb/gemini-2.0-flash/soft+cot"
print("Model directory:", load_dir)
print(*os.listdir(load_dir), sep="\n")

Model directory: /home/hgkahng/Workspaces/soft-prompt/results/imdb/gemini-2.0-flash/soft+cot
template.jsonl
embeddings
config.yaml
data.jsonl
template_formatted.txt


In [5]:
# Print configurations

with open(load_dir / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

style = "red" if cfg['hard'] else "green"

table = Table(title="Configuration(s)")
table.add_column("Name", justify="right", style="white", no_wrap=True)
table.add_column("Value", justify="left", style=style)
_ = [table.add_row(k, str(v)) for k, v in cfg.items()]

console = Console()
console.print(table)

         Configuration(s)          
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃         Name ┃ Value            ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│   batch_size │ 50               │
│          cot │ True             │
│         data │ imdb             │
│         hard │ False            │
│ log_interval │ 2                │
│   max_tokens │ 16384            │
│        model │ gemini-2.0-flash │
│ num_examples │ None             │
│  sample_size │ 50000            │
│  temperature │ 1.0              │
└──────────────┴──────────────────┘

In [6]:
with open(load_dir / "data.jsonl", "r") as f:
    data = [json.loads(line) for line in f]

print(f"Data size: {len(data):,}")

Data size: 50,017


In [7]:
data[-1]

{'label': 0.39,
 'text': '"Crimson Horizon" had potential, I\'ll give it that. The initial premise was intriguing: a sci-fi thriller set on a newly colonized planet. The visuals, especially the landscape shots, were undeniably impressive, creating a sense of alien beauty. However, the film quickly devolved into a predictable and frankly, boring, slog.\n\nThe acting was uneven at best. The lead actor seemed completely miscast, delivering his lines with all the emotional range of a brick wall. The supporting cast fared slightly better, but they were ultimately weighed down by a script that was riddled with clichés and plot holes large enough to drive a spaceship through.\n\nThe pacing was a major issue. The first act crawled along at a snail\'s pace, while the second and third acts felt rushed and disjointed. Character development was almost nonexistent, making it difficult to care about anyone on screen. By the time the credits rolled, I was left feeling disappointed and frankly, a litt

In [8]:
# labels
labels = [d['label'] for d in data]
labels = np.array(labels)

soft_labels = labels.copy()
hard_labels = (soft_labels > 0.5).astype(int)

print("Hard labels, shape:", hard_labels.shape)
print("Soft labels, shape:", soft_labels.shape)

Hard labels, shape: (50017,)
Soft labels, shape: (50017,)


In [9]:
# text
texts = [d['text'] for d in data]
len(texts)

50017

In [10]:
# embeddings
embeddings = np.load(
    load_dir / "embeddings/openai/text-embedding-3-small/data.npy"
)
print(embeddings.shape)

(50017, 1536)


In [11]:
assert len(texts) == embeddings.shape[0]
assert len(texts) == labels.shape[0]
print(len(texts))

50017


In [12]:
%%time
rng = np.random.default_rng(42)
sample_idx = rng.permutation(embeddings.shape[0])[:N_train]

labels = labels[sample_idx]
embeddings = embeddings[sample_idx]
texts = [t for i, t in enumerate(texts) if i in sample_idx]

assert len(texts) == embeddings.shape[0]
assert len(texts) == labels.shape[0]
print(len(texts))

25000
CPU times: user 304 ms, sys: 23 ms, total: 327 ms
Wall time: 326 ms


## 2. Diversity Metrics

In [13]:
from softprompt.metrics.diversity import (
    vocabulary_size,
    distinct_n,
    average_pairwise_similarity,
    average_pairwise_similarity_by_class,
    inter_sample_ngram_freq
)

In [14]:
vocab_size = vocabulary_size(texts)
print(f"Vocab: {vocab_size:>7,}")

Vocab:  13,843


In [15]:
distinct_2 = distinct_n(texts, n=2)
print(f"Distinct-2: {distinct_2:.4f}")

Distinct-2: 0.0271


In [16]:
%%time
inter_sim, intra_sim = average_pairwise_similarity_by_class(
    embeddings, labels
)
print(f"Inter-class APS: {inter_sim:.4f}\n",
      f"Intra-class APS: {intra_sim:.4f}")

Inter-class APS: 0.6282
 Intra-class APS: 0.7488
CPU times: user 49.8 s, sys: 29.3 s, total: 1min 19s
Wall time: 4.28 s


In [17]:
%%time
avg_ps = average_pairwise_similarity(embeddings)
print(f"Avg Pairwise Similarity: {avg_ps:.4f}")

Avg Pairwise Similarity: 0.6284
CPU times: user 45.2 s, sys: 23.4 s, total: 1min 8s
Wall time: 4.59 s
